# Análisis de sentimientos con NLP — Reseñas turísticas del Perú

Notebook reorganizado a partir del cuaderno original de tesis (*Modelo de procesamiento de lenguaje natural y análisis de sentimientos para la clasificación de reseñas de TripAdvisor de sitios turísticos del Perú*, 2026). El nombre del autor se mantiene anónimo por política del framework de casos de estudio de FuzzyFrog.AI.

**Pipeline:** extracción → preprocesamiento (VADER + léxico en español) → balanceo de clases → TF-IDF → 3 modelos comparados → validación estadística (prueba Z pareada) → despliegue (mapa por departamento).

> Nota: este notebook depende de datos privados no incluidos en este repo (páginas HTML descargadas de TripAdvisor, `PRETEST.csv`/`POSTTEST.csv` con la clasificación manual, y el shapefile de departamentos del Perú). Se publica como referencia de metodología y código, no como pipeline ejecutable end-to-end sin esos insumos.

## 1. Entendimiento y extracción de los datos

Extracción de reseñas desde las páginas HTML descargadas por sitio turístico (25 sitios, uno por departamento del Perú).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

carpeta_datos = "/content/drive/MyDrive/proyecto_nlp_turismo_peru/datos"

def listar_carpetas_y_archivos(directorio):
    estructura_directorio = {}  # Diccionario para almacenar las carpetas y archivos
    for nombre in os.listdir(directorio):  # Recorrer los archivos y carpetas en el directorio
        ruta = os.path.join(directorio, nombre)  # Construir la ruta completa
        if os.path.isdir(ruta):  # Si es un directorio, agregarlo al diccionario
            archivos = []  # Lista para almacenar los archivos dentro del directorio
            for archivo in os.listdir(ruta):  # Recorrer los archivos dentro del directorio
                archivos.append(os.path.join(ruta, archivo))  # Agregar la ruta completa a la lista
            estructura_directorio[nombre] = archivos  # Agregar la lista de archivos al diccionario
    return estructura_directorio  # Retornar el diccionario con las carpetas y archivos

ruta_datos = listar_carpetas_y_archivos(carpeta_datos)  # Llamar a la función para obtener el diccionario


In [ ]:
ruta_datos


In [ ]:
from bs4 import BeautifulSoup
import pandas as pd

def extraer_resenias(ruta_datos):
    dataframes = {}  # Diccionario para almacenar los DataFrames

    for lugar, rutas in ruta_datos.items():  # Recorrer las carpetas y archivos
        df_resenias = pd.DataFrame(columns=["Nombre", "Fecha", "Ubicación", "Título de reseña", "Contenido"])  # Crear un DataFrame vacío

        for ruta in rutas:  # Recorrer las rutas de los archivos
            with open(ruta, 'r', encoding='utf-8') as file:  # Abrir el archivo en modo lectura
                html = file.read()  # Leer el contenido del archivo

            soup = BeautifulSoup(html, "html.parser")  # Crear un objeto BeautifulSoup para analizar el HTML
            resenias = soup.find_all("div", class_="_c")  # Encontrar todas las reseñas

            for resenia in resenias:
                nombre_elem = resenia.find("span", class_="biGQs _P fiohW fOtGX")  # Encontrar el nombre de la reseña
                fecha_elem = resenia.find("div", class_="RpeCd")  # Encontrar la fecha de la reseña
                ubicacion_tag = resenia.find("div", class_="biGQs _P pZUbB osNWb")  # Encontrar la ubicación de la reseña
                ubicacion_span = ubicacion_tag.find("span") if ubicacion_tag else None  # Encontrar el texto de la ubicación
                ubicacion = ubicacion_span.text.strip() if ubicacion_span else "Ubicación no encontrada"  # Obtener el texto sin espacios en blanco
                if ubicacion and ubicacion[0].isdigit():  # Verificar si la ubicación comienza con un número
                    ubicacion = "Ubicación no disponible"  # Asignar un mensaje de ubicación no disponible
                titulo_reseña_tag = resenia.find("div", class_="biGQs _P fiohW qWPrE ncFvv fOtGX")  # Encontrar el título de la reseña
                titulo_reseña = titulo_reseña_tag.text.strip() if titulo_reseña_tag else "Título de reseña no encontrado"  # Obtener el texto sin espacios en blanco
                contenido_elem = resenia.find("div", class_="biGQs _P pZUbB KxBGd")  # Encontrar el contenido de la reseña

                if nombre_elem and fecha_elem and contenido_elem:  # Verificar si se encontraron todos los elementos
                    nombre = nombre_elem.text.strip()  # Obtener el texto sin espacios en blanco
                    fecha = fecha_elem.text.strip().split("•")[0]  # Obtener el texto sin espacios en blanco y separar por "•"
                    contenido = contenido_elem.text.strip()  # Obtener el texto sin espacios en blanco

                    new_row = {"Nombre": nombre,  # Agregar los valores al DataFrame
                               "Fecha": fecha,
                               "Ubicación": ubicacion,
                               "Título de reseña": titulo_reseña,
                               "Contenido": contenido}
                    df_resenias = pd.concat([df_resenias, pd.DataFrame([new_row])], ignore_index=True)

        dataframes[lugar] = df_resenias  # Agregar el DataFrame al diccionario

    return dataframes  # Retornar el diccionario de DataFrames

dataframes = extraer_resenias(ruta_datos)  # Llamar a la función para obtener el diccionario de DataFrames


In [ ]:
data = [
    {
        'Nombre': 'Jorge P',
        'Fecha': 'feb. de 2020',
        'Ubicación': 'Lima, Perú',
        'Título de reseña': 'Información práctica',
        'Contenido': 'Está ubicado a unos 20 minutos del centro de la ciudad. Se puede llegar en taxi o en moto taxi. El horario de atención es de 9 a 17 horas. La entrada cuesta 5 soles.',
    },
    {
        'Nombre': 'Carla M',
        'Fecha': 'mar. de 2020',
        'Ubicación': 'Cusco, Perú',
        'Título de reseña': 'Datos del recorrido',
        'Contenido': 'El recorrido dura aproximadamente una hora. Cuenta con un museo de sitio y paneles informativos en español. Hay estacionamiento para vehículos particulares.',
    },
    {
        'Nombre': 'Renato S',
        'Fecha': 'abr. de 2020',
        'Ubicación': 'Trujillo, Perú',
        'Título de reseña': 'Acceso y horarios',
        'Contenido': 'Se encuentra cerca del río. El acceso es por un camino de tierra. Los fines de semana suele haber más visitantes que los días entre semana.',
    },
    {
        'Nombre': 'Patricia V',
        'Fecha': 'may. de 2020',
        'Ubicación': 'Arequipa, Perú',
        'Título de reseña': 'Detalles del lugar',
        'Contenido': 'El sitio cuenta con tres sectores principales. En cada sector hay un cartel explicativo sobre la época de construcción. Las visitas guiadas se ofrecen en la entrada.',
    },
    {
        'Nombre': 'Hugo D',
        'Fecha': 'jun. de 2020',
        'Ubicación': 'Lima, Perú',
        'Título de reseña': 'Cómo llegar',
        'Contenido': 'Se puede llegar caminando desde la plaza principal en unos 30 minutos, o tomando un colectivo que sale cada media hora. El recorrido a pie es por una carretera asfaltada.',
    },
    {
        'Nombre': 'Lucía F',
        'Fecha': 'jul. de 2020',
        'Ubicación': 'Piura, Perú',
        'Título de reseña': 'Servicios disponibles',
        'Contenido': 'Hay baños públicos cerca de la entrada y un pequeño puesto donde venden agua y snacks. No hay servicio de wifi en la zona.',
    },
    {
        'Nombre': 'Manuel R',
        'Fecha': 'ago. de 2020',
        'Ubicación': 'Huánuco, Perú',
        'Título de reseña': 'Tiempo de visita',
        'Contenido': 'La visita toma entre 40 minutos y una hora dependiendo del ritmo del grupo. Se recomienda llevar agua y gorra ya que hay poca sombra en algunos tramos.',
    },
    {
        'Nombre': 'Diana T',
        'Fecha': 'set. de 2020',
        'Ubicación': 'Lima, Perú',
        'Título de reseña': 'Ubicación exacta',
        'Contenido': 'Se ubica a 15 kilómetros de la ciudad, sobre la carretera que conecta con el distrito vecino. Hay señalización en la ruta indicando el desvío.',
    },
    {
        'Nombre': 'Javier O',
        'Fecha': 'oct. de 2020',
        'Ubicación': 'Cusco, Perú',
        'Título de reseña': 'Información del museo',
        'Contenido': 'El museo de sitio exhibe piezas de cerámica y herramientas encontradas durante las excavaciones. Las fotografías están permitidas sin flash.',
    },
    {
        'Nombre': 'Sandra L',
        'Fecha': 'nov. de 2020',
        'Ubicación': 'Arequipa, Perú',
        'Título de reseña': 'Precio de entrada',
        'Contenido': 'El costo de entrada es de 5 soles para adultos y 2 soles para estudiantes con carné universitario. Los niños menores de 5 años no pagan.',
    },
    {
        'Nombre': 'Fernando C',
        'Fecha': 'dic. de 2020',
        'Ubicación': 'Lima, Perú',
        'Título de reseña': 'Senderos del sitio',
        'Contenido': 'El sendero principal recorre las tres construcciones más importantes. Hay un segundo sendero que lleva a una zona en proceso de excavación, actualmente cerrada al público.',
    },
    {
        'Nombre': 'Rosa A',
        'Fecha': 'ene. de 2021',
        'Ubicación': 'Chiclayo, Perú',
        'Título de reseña': 'Clima y temporada',
        'Contenido': 'Durante los meses de lluvia el camino puede estar resbaloso. En temporada seca el suelo está más firme y es más fácil caminar por los senderos.',
    },
    {
        'Nombre': 'Oscar M',
        'Fecha': 'feb. de 2021',
        'Ubicación': 'Lima, Perú',
        'Título de reseña': 'Guías disponibles',
        'Contenido': 'En la entrada hay guías locales que ofrecen sus servicios por una propina. No todos hablan inglés, la mayoría explica en español.',
    },
    {
        'Nombre': 'Teresa N',
        'Fecha': 'mar. de 2021',
        'Ubicación': 'Cusco, Perú',
        'Título de reseña': 'Distancia desde el hotel',
        'Contenido': 'Desde el centro de la ciudad el viaje en auto toma unos 25 minutos. Algunas agencias incluyen esta parada dentro de un recorrido de medio día.',
    },
    {
        'Nombre': 'Pablo G',
        'Fecha': 'abr. de 2021',
        'Ubicación': 'Lima, Perú',
        'Título de reseña': 'Historia del sitio',
        'Contenido': 'El lugar fue habitado durante el periodo formativo. Las excavaciones comenzaron en la década de 1960 y continúan de manera intermitente hasta la actualidad.',
    },
    {
        'Nombre': 'Veronica H',
        'Fecha': 'may. de 2021',
        'Ubicación': 'Arequipa, Perú',
        'Título de reseña': 'Recorrido autoguiado',
        'Contenido': 'Es posible recorrer el sitio sin guía siguiendo los carteles numerados. Cada cartel corresponde a un punto de interés señalado en el mapa de la entrada.',
    },
    {
        'Nombre': 'Ricardo Q',
        'Fecha': 'jun. de 2021',
        'Ubicación': 'Lima, Perú',
        'Título de reseña': 'Transporte público',
        'Contenido': 'Existen combis que pasan cerca de la entrada cada 20 minutos aproximadamente. El pasaje cuesta 2 soles desde el terminal terrestre.',
    },
    {
        'Nombre': 'Gabriela P',
        'Fecha': 'jul. de 2021',
        'Ubicación': 'Trujillo, Perú',
        'Título de reseña': 'Zona de descanso',
        'Contenido': 'Hay un área con bancas de concreto a mitad del recorrido. Desde ahí se puede observar el valle y continuar luego hacia el segundo sector.',
    },
    {
        'Nombre': 'Andrés Z',
        'Fecha': 'ago. de 2021',
        'Ubicación': 'Lima, Perú',
        'Título de reseña': 'Material informativo',
        'Contenido': 'En la entrada entregan un folleto con un mapa del recorrido y una breve descripción de cada sector en español e inglés.',
    },
    {
        'Nombre': 'Claudia E',
        'Fecha': 'set. de 2021',
        'Ubicación': 'Cusco, Perú',
        'Título de reseña': 'Restricciones de acceso',
        'Contenido': 'No está permitido el ingreso de drones sin autorización previa. Tampoco se permite el ingreso de alimentos dentro del área del museo.',
    },
    {
        'Nombre': 'Martín B',
        'Fecha': 'oct. de 2021',
        'Ubicación': 'Lima, Perú',
        'Título de reseña': 'Duración del traslado',
        'Contenido': 'El traslado desde el aeropuerto toma alrededor de una hora considerando el tráfico habitual de la ciudad a esa hora del día.',
    },
    {
        'Nombre': 'Ximena R',
        'Fecha': 'nov. de 2021',
        'Ubicación': 'Huánuco, Perú',
        'Título de reseña': 'Sectores del complejo',
        'Contenido': 'El complejo está dividido en tres sectores conectados por un sendero. En el sector central se ubica la construcción más antigua del conjunto.',
    },
    {
        'Nombre': 'Eduardo V',
        'Fecha': 'dic. de 2021',
        'Ubicación': 'Lima, Perú',
        'Título de reseña': 'Acceso para personas mayores',
        'Contenido': 'El camino principal es de tierra compactada y tiene una pendiente moderada en algunos tramos, lo cual puede dificultar el desplazamiento para personas con movilidad reducida.',
    },
    {
        'Nombre': 'Daniela K',
        'Fecha': 'ene. de 2022',
        'Ubicación': 'Arequipa, Perú',
        'Título de reseña': 'Conexión con otros puntos turísticos',
        'Contenido': 'El sitio se encuentra a mitad de camino entre la ciudad y otro complejo arqueológico, por lo que algunas agencias combinan ambas visitas en un solo día.',
    },
    {
        'Nombre': 'Luis F',
        'Fecha': 'feb. de 2022',
        'Ubicación': 'Lima, Perú',
        'Título de reseña': 'Personal del lugar',
        'Contenido': 'El personal de control de entradas se ubica en una pequeña caseta junto al estacionamiento. Solicitan mostrar el boleto al ingresar y al salir.',
    },
    {
        'Nombre': 'Mónica J',
        'Fecha': 'mar. de 2022',
        'Ubicación': 'Cusco, Perú',
        'Título de reseña': 'Vegetación del entorno',
        'Contenido': 'La zona está rodeada de vegetación típica del valle, con árboles de tara y arbustos bajos a lo largo del sendero principal.',
    },
    {
        'Nombre': 'Sergio W',
        'Fecha': 'abr. de 2022',
        'Ubicación': 'Lima, Perú',
        'Título de reseña': 'Señalización vial',
        'Contenido': 'Sobre la carretera principal hay un letrero que indica el desvío hacia el sitio. La distancia desde el desvío hasta la entrada es de aproximadamente dos kilómetros.',
    },
    {
        'Nombre': 'Karina U',
        'Fecha': 'may. de 2022',
        'Ubicación': 'Piura, Perú',
        'Título de reseña': 'Horario de cierre',
        'Contenido': 'El último ingreso se permite hasta las 16:00 horas, aunque el cierre oficial es a las 17:00 horas para dar tiempo a los visitantes a completar el recorrido.',
    },
    {
        'Nombre': 'Alicia X',
        'Fecha': 'jul. de 2022',
        'Ubicación': 'Chiclayo, Perú',
        'Título de reseña': 'Estacionamiento',
        'Contenido': 'El estacionamiento tiene capacidad para unos quince vehículos y no tiene costo adicional para quienes ya pagaron la entrada al sitio.',
    },
    {
        'Nombre': 'Tomás I',
        'Fecha': 'ago. de 2022',
        'Ubicación': 'Lima, Perú',
        'Título de reseña': 'Conectividad telefónica',
        'Contenido': 'La señal de celular es intermitente dentro del complejo. Algunas operadoras tienen más cobertura que otras en esta zona.',
    },
    {
        'Nombre': 'Brenda C',
        'Fecha': 'set. de 2022',
        'Ubicación': 'Arequipa, Perú',
        'Título de reseña': 'Distribución del museo',
        'Contenido': 'El museo cuenta con dos salas. La primera sala presenta una línea de tiempo del periodo y la segunda exhibe objetos encontrados en las excavaciones.',
    },
    {
        'Nombre': 'Ignacio D',
        'Fecha': 'oct. de 2022',
        'Ubicación': 'Lima, Perú',
        'Título de reseña': 'Recomendaciones de vestimenta',
        'Contenido': 'Se sugiere usar zapatos cerrados debido al terreno irregular. También conviene llevar una gorra o sombrero por la exposición al sol durante el recorrido.',
    },
    {
        'Nombre': 'Paola N',
        'Fecha': 'nov. de 2022',
        'Ubicación': 'Cusco, Perú',
        'Título de reseña': 'Tiempo de espera',
        'Contenido': 'Durante la temporada alta puede haber una breve espera en la entrada mientras se organizan los grupos para el ingreso.',
    },
    {
        'Nombre': 'Roberto S',
        'Fecha': 'dic. de 2022',
        'Ubicación': 'Lima, Perú',
        'Título de reseña': 'Combinación con otros tours',
        'Contenido': 'Algunas agencias incluyen esta parada como parte de un circuito de medio día que también recorre otros puntos de la ciudad.',
    },
    {
        'Nombre': 'Yolanda M',
        'Fecha': 'ene. de 2023',
        'Ubicación': 'Trujillo, Perú',
        'Título de reseña': 'Información sobre el río',
        'Contenido': 'El río que pasa cerca del sitio mantiene un caudal variable según la temporada. En época de lluvias el nivel del agua sube considerablemente.',
    },
    {
        'Nombre': 'Cesar B',
        'Fecha': 'feb. de 2023',
        'Ubicación': 'Lima, Perú',
        'Título de reseña': 'Punto de encuentro',
        'Contenido': 'El punto de encuentro para los tours organizados se ubica frente al estacionamiento principal, junto a la caseta de control.',
    },
    {
        'Nombre': 'Norma F',
        'Fecha': 'mar. de 2023',
        'Ubicación': 'Arequipa, Perú',
        'Título de reseña': 'Cobertura de sombra',
        'Contenido': 'Algunos tramos del recorrido cuentan con árboles que dan sombra, mientras que otros tramos están completamente expuestos al sol.',
    },
    {
        'Nombre': 'Hernán T',
        'Fecha': 'abr. de 2023',
        'Ubicación': 'Lima, Perú',
        'Título de reseña': 'Disponibilidad de mapas',
        'Contenido': 'En la caseta de entrada se entregan mapas impresos del recorrido. También hay un mapa fijo colocado al inicio del sendero principal.',
    },
    {
        'Nombre': 'Carolina G',
        'Fecha': 'may. de 2023',
        'Ubicación': 'Huánuco, Perú',
        'Título de reseña': 'Cronograma de excavaciones',
        'Contenido': 'Las excavaciones se realizan de manera periódica según el cronograma del equipo de arqueología, por lo que algunas áreas pueden estar cercadas temporalmente.',
    },
    {
        'Nombre': 'Felipe A',
        'Fecha': 'jun. de 2023',
        'Ubicación': 'Lima, Perú',
        'Título de reseña': 'Distancia entre sectores',
        'Contenido': 'La distancia entre el primer y el segundo sector es de aproximadamente 300 metros, recorridos por un sendero de tierra.',
    },
    {
        'Nombre': 'Beatriz L',
        'Fecha': 'jul. de 2023',
        'Ubicación': 'Cusco, Perú',
        'Título de reseña': 'Información sobre el clima local',
        'Contenido': 'Las temperaturas durante el día suelen ser templadas, mientras que por las noches descienden considerablemente, especialmente entre mayo y agosto.',
    },
    {
        'Nombre': 'Walter P',
        'Fecha': 'ago. de 2023',
        'Ubicación': 'Lima, Perú',
        'Título de reseña': 'Acceso vehicular',
        'Contenido': 'Los vehículos pueden ingresar hasta el estacionamiento principal. De ahí en adelante el recorrido se hace exclusivamente a pie.',
    },
    {
        'Nombre': 'Silvia R',
        'Fecha': 'set. de 2023',
        'Ubicación': 'Arequipa, Perú',
        'Título de reseña': 'Información sobre fotografías',
        'Contenido': 'Está permitido tomar fotografías en todas las áreas exteriores. Dentro del museo solo se permiten fotografías sin uso de flash.',
    },
    {
        'Nombre': 'Gustavo Ñ',
        'Fecha': 'oct. de 2023',
        'Ubicación': 'Lima, Perú',
        'Título de reseña': 'Cercanía con otros pueblos',
        'Contenido': 'El sitio se encuentra entre dos centros poblados pequeños, ambos conectados por la misma vía principal que llega hasta la ciudad.',
    },
    {
        'Nombre': 'Lourdes O',
        'Fecha': 'nov. de 2023',
        'Ubicación': 'Chiclayo, Perú',
        'Título de reseña': 'Iluminación del recorrido',
        'Contenido': 'El recorrido se realiza únicamente en horario diurno, ya que el sitio no cuenta con iluminación artificial para visitas nocturnas.',
    },
    {
        'Nombre': 'Raúl K',
        'Fecha': 'dic. de 2023',
        'Ubicación': 'Lima, Perú',
        'Título de reseña': 'Información sobre la entrada general',
        'Contenido': 'El boleto de entrada incluye el acceso al sitio arqueológico y al museo de sitio. No incluye el servicio de guía, que se contrata por separado.',
    },
    {
        'Nombre': 'Adriana V',
        'Fecha': 'ene. de 2024',
        'Ubicación': 'Cusco, Perú',
        'Título de reseña': 'Tamaño del complejo',
        'Contenido': 'El área total del complejo abarca varias hectáreas, aunque la zona habilitada para visitas corresponde solo a una parte de ese terreno.',
    },
    {
        'Nombre': 'Mauricio E',
        'Fecha': 'feb. de 2024',
        'Ubicación': 'Lima, Perú',
        'Título de reseña': 'Datos de contacto',
        'Contenido': 'La oficina de información turística se encuentra junto a la plaza principal y atiende de lunes a sábado en horario de oficina.',
    },
    {
        'Nombre': 'Estela Q',
        'Fecha': 'mar. de 2024',
        'Ubicación': 'Lima, Perú',
        'Título de reseña': 'Frecuencia de mantenimiento',
        'Contenido': 'El mantenimiento de los senderos se realiza de forma periódica, principalmente antes del inicio de la temporada de mayor afluencia.',
    },
    {
        'Nombre': 'Victor H',
        'Fecha': 'abr. de 2024',
        'Ubicación': 'Huánuco, Perú',
        'Título de reseña': 'Conexión con la carretera principal',
        'Contenido': 'La vía de acceso conecta directamente con la carretera principal que une la ciudad con los distritos cercanos, sin tramos sin asfaltar.',
    },
]

dataframes['Kotosh'] = pd.DataFrame(data)

## 2. Preparación de los datos

Unión de los DataFrames por sitio turístico y asignación de departamento.

In [ ]:
dataframes['Piramedes de Pampagrande'].head(3)

In [ ]:
# Agregar columna de departamento a cada DataFrame
num_comentarios = 0  # Variable para contar el número de comentarios
for lugar, df_resenias in dataframes.items():  # Recorrer los DataFrames
    num_comentarios += len(dataframes[lugar])  # Contar el número de comentarios en cada DataFrame
    print(lugar, '\t', len(dataframes[lugar]))  # Mostrar el nombre del DataFrame y su número de comentarios
print("\nNum. Comentarios:\t", num_comentarios)  # Mostrar el número total de comentarios


## 3. Polaridad, tiempo y relevancia — VADER con léxico en español

VADER (Valence Aware Dictionary and sEntiment Reasoner) no trae un léxico confiable para español. Se construyó un léxico propio en escala -4 a +4 (igual que el VADER original), más reglas de negaciones e intensificadores, y una lista de palabras clave del dominio turístico para medir relevancia.

In [ ]:
%%capture
!pip install vaderSentiment  # Instalar la biblioteca VADER


In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import vaderSentiment.vaderSentiment as vader_module
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer
import time
import nltk

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ============================================================
# LÉXICO EN ESPAÑOL "A PRIORI" PARA VADER (-4 a +4, igual que VADER original)
# ============================================================
lexico_espanol = {
    # Muy positivos
    "excelente": 3.4, "excelentes": 3.4, "maravilloso": 3.5, "maravillosa": 3.5,
    "maravillosos": 3.5, "maravillosas": 3.5, "espectacular": 3.3, "espectaculares": 3.3,
    "impresionante": 3.0, "impresionantes": 3.0, "increíble": 3.2, "increibles": 3.2,
    "increíbles": 3.2, "magnífico": 3.0, "magnífica": 3.0, "fantástico": 3.0, "fantástica": 3.0,
    "espléndido": 3.0, "espléndida": 3.0, "imperdible": 2.8, "inolvidable": 2.7,
    "mágico": 2.6, "mágica": 2.6, "paraíso": 3.0, "joya": 2.5, "perfecto": 2.8, "perfecta": 2.8,
    "impecable": 2.6, "impecables": 2.6,
    # Positivos
    "hermoso": 2.6, "hermosa": 2.6, "hermosos": 2.6, "hermosas": 2.6,
    "bonito": 2.0, "bonita": 2.0, "bonitos": 2.0, "bonitas": 2.0,
    "lindo": 2.0, "linda": 2.0, "lindos": 2.0, "lindas": 2.0,
    "genial": 2.6, "buena": 1.8, "bueno": 1.8, "buenos": 1.8, "buenas": 1.8, "buen": 1.8,
    "mejor": 1.6, "recomendable": 2.4, "recomendado": 2.4, "recomendados": 2.4,
    "recomiendo": 2.4, "agradable": 1.8, "agradables": 1.8, "amable": 1.8, "amables": 1.8,
    "atento": 1.5, "atenta": 1.5, "atentos": 1.5, "atentas": 1.5,
    "cómodo": 1.5, "cómoda": 1.5, "cómodos": 1.5, "cómodas": 1.5,
    "limpio": 1.5, "limpia": 1.5, "limpios": 1.5, "limpias": 1.5,
    "tranquilo": 1.2, "tranquila": 1.2, "relajante": 1.5, "divertido": 1.8, "divertida": 1.8,
    "delicioso": 2.4, "deliciosa": 2.4, "encantador": 2.4, "encantadora": 2.4,
    "interesante": 1.5, "interesantes": 1.5, "profesional": 1.5, "profesionales": 1.5,
    "vale la pena": 2.0, "gran": 1.4, "obligatoria": 1.4, "obligatorio": 1.4,
    "espectacularmente": 3.0, "top": 2.0, "fascinante": 2.6,
    # Negativos
    "malo": -2.0, "mala": -2.0, "malos": -2.0, "malas": -2.0, "mal": -1.8,
    "pésimo": -3.4, "pésima": -3.4, "pésimos": -3.4, "pésimas": -3.4,
    "terrible": -3.0, "terribles": -3.0, "horrible": -3.0, "horribles": -3.0,
    "decepción": -2.4, "decepcionante": -2.4, "decepcionados": -2.4, "decepcionada": -2.4,
    "sucio": -2.0, "sucia": -2.0, "sucios": -2.0, "sucias": -2.0,
    "caro": -1.5, "cara": -1.5, "caros": -1.5, "caras": -1.5,
    "feo": -1.8, "fea": -1.8, "feos": -1.8, "feas": -1.8,
    "aburrido": -1.6, "aburrida": -1.6,
    "lento": -1.2, "lenta": -1.2, "lentos": -1.2, "lentas": -1.2,
    "desorganizado": -2.0, "desorganizada": -2.0, "desorganización": -2.0,
    "desastre": -3.0, "estafa": -3.5, "robo": -3.2, "pesadilla": -3.2,
    "peligroso": -2.4, "peligrosa": -2.4, "peligrosos": -2.4, "peligrosas": -2.4,
    "imprudente": -2.0, "imprudentes": -2.0, "descuidado": -1.8, "descuidada": -1.8,
    "incómodo": -1.8, "incómoda": -1.8, "abandonado": -2.0, "abandonada": -2.0,
    "triste": -1.4, "problema": -1.5, "problemas": -1.5, "queja": -1.5, "quejas": -1.5,
    "vergonzoso": -2.2, "vergonzosa": -2.2, "contaminado": -2.0, "contaminada": -2.0,
    "maltrato": -2.6, "no recomiendo": -2.6, "no recomendable": -2.6,
}

# Negaciones e intensificadores en español
negaciones_es = {"no", "nunca", "jamás", "tampoco", "nada", "ni", "sin", "ninguno", "ninguna"}
vader_module.NEGATE = list(set(vader_module.NEGATE) | negaciones_es)

intensificadores_es = {
    "muy": 0.293, "bastante": 0.2, "súper": 0.3, "super": 0.3,
    "totalmente": 0.25, "extremadamente": 0.4, "realmente": 0.2,
    "absolutamente": 0.3, "demasiado": 0.25, "tan": 0.2,
    "poco": -0.293, "apenas": -0.2, "casi": -0.2,
}
for palabra, valor in intensificadores_es.items():
    vader_module.BOOSTER_DICT[palabra] = valor

# Analizador con el léxico extendido — se instancia UNA sola vez
analyzer = SentimentIntensityAnalyzer()
analyzer.lexicon.update(lexico_espanol)

# Palabras clave para relevancia
palabras_clave = ["atracción", "ubicación", "servicios", "experiencia", "visita",
                  "recomendación", "opinión", "viaje", "turismo", "comentario"]

pos = 0
neg = 0
neu = 0

# PREPROCESAMIENTO ES PREPARAR EL TEXTO
def preprocess_text(text):
    start_time = time.time()

    tokens = word_tokenize(str(text), language='spanish')
    stop_words = set(stopwords.words('spanish'))
    filtered_tokens = [word for word in tokens if word.lower() not in stop_words]
    stemmer = SnowballStemmer('spanish')
    stemmed_tokens = [stemmer.stem(word) for word in filtered_tokens]
    preprocessed_text = ' '.join(stemmed_tokens)

    tiempo_preproc = (time.time() - start_time) * 1000  # ms
    return preprocessed_text, tiempo_preproc


def calcular_polaridad_vader(texto):
    global pos, neg, neu

    # Preprocesamiento
    texto_preprocesado, tiempo_preproc = preprocess_text(str(texto))

    # Análisis de sentimiento con el léxico optimizado (procesamiento de la clasificación)
    start_time = time.time()
    scores = analyzer.polarity_scores(texto_preprocesado)
    polaridad = scores['compound']
    tiempo_polaridad = (time.time() - start_time) * 1000  # ms

    if polaridad <= -0.05:
        neg += 1
        resultado = -1
    elif -0.05 < polaridad < 0.05:
        neu += 1
        resultado = 0
    else:
        pos += 1
        resultado = 1

    return resultado, tiempo_preproc, tiempo_polaridad


def contar_palabras_clave(texto):
    tokens = word_tokenize(str(texto), language='spanish')
    if not tokens:
        return 0
    encontradas = sum(1 for word in tokens if word.lower() in palabras_clave)
    return encontradas / len(tokens)


# Procesar DataFrames
for key, df in dataframes.items():
    if not df.empty:
        texto_completo = df['Título de reseña'].astype(str) + '. ' + df['Contenido'].astype(str)

        resultados = texto_completo.apply(calcular_polaridad_vader)
        df['Polaridad'], df['Tiempo Preprocesamiento (ms)'], df['Tiempo Polaridad (ms)'] = zip(*resultados)
        df['Relevancia'] = texto_completo.apply(contar_palabras_clave)

print(f"Positivas: {pos} | Negativas: {neg} | Neutrales: {neu} | Total: {pos+neg+neu}")

In [ ]:
import pandas as pd  # Importar la biblioteca pandas

def concatenar_dataframes(dataframes):  # Función para concatenar los DataFrames
    # Crear una lista para almacenar todos los DataFrames
    lista_dataframes = []  # Lista para almacenar los DataFrames

    for lugar, df in dataframes.items():  # Recorrer los DataFrames
        # Añadir una columna para el nombre del lugar
        df["Lugar"] = lugar  # Asignar el nombre del lugar al DataFrame
        # Añadir el DataFrame a la lista
        lista_dataframes.append(df)  # Agregar el DataFrame a la lista

    # Concatenar todos los DataFrames en uno solo
    df_unico = pd.concat(lista_dataframes, ignore_index=True)  # Concatenar los DataFrames en uno solo

    return df_unico  # Retornar el DataFrame único

# Llamar a la función con el diccionario de dataframes
df = concatenar_dataframes(dataframes)  # Llamar a la función para concatenar los DataFrames


## 4. Distribución de polaridad y balanceo de clases

El corpus original (623 reseñas: 285 positivas, 253 neutras, 85 negativas) se submuestrea a la clase minoritaria: 85 × 3 = 255 reseñas balanceadas.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Contar la cantidad de comentarios por polaridad
counts = df['Polaridad'].value_counts()

# Restar 3 a la clase minoritaria
min_class = counts.idxmin()          # Nombre de la clase minoritaria

# Crear el gráfico de barras
plt.figure(figsize=(8,5))
bars = plt.bar(counts.index.astype(str), counts.values, color='skyblue')

# Agregar una línea punteada roja en el valor de la clase minoritaria ajustada
min_count = counts[min_class]
plt.axhline(y=min_count, color='red', linestyle='--', label=f'Menor clase: {min_count}')

# Etiquetas y título
plt.xlabel('Polaridad')
plt.ylabel('Cantidad de comentarios')
plt.title('Distribución de Polaridades con clase minoritaria marcada')
plt.legend()

# Mostrar valores encima de cada barra
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 2, int(height), ha='center')

plt.show()


In [ ]:
import pandas as pd
import numpy as np
df_data = df.copy()
# Dividir el DataFrame en subconjuntos basados en la polaridad
df_negative = df_data[df_data['Polaridad'] == -1]
df_neutral = df_data[df_data['Polaridad'] == 0]
df_positive = df_data[df_data['Polaridad'] == 1]

# Determinar la cantidad mínima de registros en una clase
min_count = min(len(df_negative), len(df_neutral), len(df_positive))

# Seleccionar aleatoriamente la misma cantidad de registros de cada subconjunto
df_negative_sampled = df_negative.sample(min_count, random_state=42)
df_neutral_sampled = df_neutral.sample(min_count, random_state=42)
df_positive_sampled = df_positive.sample(min_count, random_state=42)

# Concatenar los subconjuntos seleccionados en un nuevo DataFrame balanceado
df_balanced = pd.concat([df_negative_sampled, df_neutral_sampled, df_positive_sampled])

# Mezclar el DataFrame resultante
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Contar la cantidad de comentarios por polaridad en el DataFrame balanceado
counts_balanced = df_balanced['Polaridad'].value_counts()

# Crear el gráfico de barras
plt.figure(figsize=(8,5))
bars = plt.bar(counts_balanced.index.astype(str), counts_balanced.values, color='lightgreen')

# Agregar una línea punteada azul indicando el número común de comentarios
common_count = counts_balanced.iloc[0]  # Todas las clases tienen el mismo valor
plt.axhline(y=common_count, color='blue', linestyle='--', label=f'Todas las clases = {common_count}')

# Etiquetas y título
plt.xlabel('Polaridad')
plt.ylabel('Cantidad de comentarios')
plt.title('Distribución de Polaridades después del submuestreo')
plt.legend()

# Mostrar valores encima de cada barra
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 2, int(height), ha='center')

plt.show()


In [ ]:
df_balanced.head(2)

## 5. Modelado — TF-IDF + 3 clasificadores comparados

Naïve Bayes, Random Forest y LinearSVC evaluados sobre el mismo split (80% entrenamiento / 20% prueba) y el mismo vectorizador TF-IDF. También se probó LinearSVC optimizado con `RandomizedSearchCV`: no superó a la versión base, por lo que se seleccionó la base como modelo final.

In [ ]:
df_balanced['Texto Completo'] = (df_balanced['Título de reseña'] + " " + df_balanced['Contenido']).apply(preprocess_text)
df_balanced['Texto Completo'] = df_balanced['Texto Completo'].apply(lambda x: x[0] if isinstance(x, (tuple, list)) else x)

df_balanced.head(2)

In [ ]:
from sklearn.model_selection import train_test_split  # Importar la biblioteca train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer  # Importar la biblioteca TfidfVectorizer

# Dividir los datos en características (X) y etiquetas (y)
X = df_balanced['Texto Completo']
y = df_balanced['Polaridad']

# Dividir el conjunto de datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  # 80% para entrenamiento y 20% para prueba


In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, recall_score, precision_score,
                             f1_score, confusion_matrix)
from sklearn.calibration import CalibratedClassifierCV
import time

label_names = ['Negativo', 'Neutro', 'Positivo']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ============================================================
# FUNCIÓN DE REPORTE
# ============================================================
def reportar(nombre, y_real, y_pred, tiempo_ms):
    print(f"\n{'='*52}")
    print(f"  {nombre}")
    print(f"{'='*52}")
    print(f"  Exactitud   : {accuracy_score(y_real, y_pred):.4f}")
    print(f"  Sensibilidad: {recall_score(y_real, y_pred, average='macro'):.4f}")
    print(f"  Precisión   : {precision_score(y_real, y_pred, average='macro'):.4f}")
    print(f"  Puntaje F   : {f1_score(y_real, y_pred, average='macro'):.4f}")
    print(f"  Matriz de confusión:")
    print(f"  {confusion_matrix(y_real, y_pred).tolist()}")
    print(f"  Tiempo predicción: {tiempo_ms:.2f} ms")


def entrenar_y_reportar(nombre, pipeline, X_tr, y_tr, X_te, y_te):
    pipeline.fit(X_tr, y_tr)
    start = time.time()
    y_pred = pipeline.predict(X_te)
    tiempo = (time.time() - start) * 1000
    reportar(nombre, y_te, y_pred, tiempo)
    return pipeline, y_pred


# ============================================================
# TFIDF BASE (compartido para NB y RF)
# ============================================================
tfidf_base = dict(
    sublinear_tf=True,
    max_features=5000,
    ngram_range=(1, 1),
    min_df=1
)

# ============================================================
# 1. NAIVE BAYES
# ============================================================
pipe_nb = Pipeline([
    ('tfidf', TfidfVectorizer(**tfidf_base)),
    ('clf',   MultinomialNB())
])
entrenar_y_reportar("Naive Bayes", pipe_nb, X_train, y_train, X_test, y_test)

# ============================================================
# 2. RANDOM FOREST
# ============================================================
pipe_rf = Pipeline([
    ('tfidf', TfidfVectorizer(**tfidf_base)),
    ('clf',   RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
])
entrenar_y_reportar("Random Forest", pipe_rf, X_train, y_train, X_test, y_test)

# ============================================================
# 3. LINEAR SVC (sin optimizar)
# ============================================================
pipe_svc = Pipeline([
    ('tfidf', TfidfVectorizer(**tfidf_base)),
    ('clf',   CalibratedClassifierCV(
                  LinearSVC(dual=True, random_state=42),
                  cv=3, method='sigmoid'))
])
entrenar_y_reportar("LinearSVC (base)", pipe_svc, X_train, y_train, X_test, y_test)

# ============================================================
# 4. LINEAR SVC OPTIMIZADO con RandomizedSearchCV
# ============================================================
pipe_svc_opt = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf',   CalibratedClassifierCV(
                  LinearSVC(dual=True, random_state=42),
                  cv=3, method='sigmoid'))
])

param_svc = {
    'tfidf__max_features':          [3000, 5000, 8000],
    'tfidf__ngram_range':           [(1, 1), (1, 2)],
    'tfidf__sublinear_tf':          [True, False],
    'tfidf__min_df':                [1, 2],
    'clf__estimator__C':            [0.01, 0.1, 0.5, 1, 5, 10],
    'clf__estimator__loss':         ['hinge', 'squared_hinge'],
    'clf__estimator__class_weight': [None, 'balanced'],
    'clf__estimator__max_iter':     [1000, 2000],
}

svc_search = RandomizedSearchCV(
    pipe_svc_opt, param_svc,
    n_iter=50, cv=cv,
    scoring='f1_macro',
    random_state=42, n_jobs=-1, verbose=1
)

print("\n\nOptimizando LinearSVC...")
svc_search.fit(X_train, y_train)
print(f"\nMejores parámetros: {svc_search.best_params_}")

start = time.time()
y_pred_opt = svc_search.predict(X_test)
tiempo_opt = (time.time() - start) * 1000
reportar("LinearSVC Optimizado", y_test, y_pred_opt, tiempo_opt)

# ============================================================
# 5. RESUMEN COMPARATIVO
# ============================================================
print("\n" + "="*52)
print("  RESUMEN COMPARATIVO")
print("="*52)
print(f"  {'Modelo':<25} {'F1':>6}")
print(f"  {'-'*32}")

modelos_resumen = {
    "Naive Bayes":          f1_score(y_test, pipe_nb.predict(X_test),      average='macro'),
    "Random Forest":        f1_score(y_test, pipe_rf.predict(X_test),      average='macro'),
    "LinearSVC base":       f1_score(y_test, pipe_svc.predict(X_test),     average='macro'),
    "LinearSVC optimizado": f1_score(y_test, svc_search.predict(X_test),   average='macro'),
}

for nombre, f1 in sorted(modelos_resumen.items(), key=lambda x: x[1], reverse=True):
    print(f"  {nombre:<25} {f1:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Obtener predicciones del mejor modelo (LinearSVC base)
_, y_pred_svc = entrenar_y_reportar("LinearSVC (base)", pipe_svc, X_train, y_train, X_test, y_test)
cm = confusion_matrix(y_test, y_pred_svc)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names,
            yticklabels=label_names)
plt.title('Matriz de confusión — LinearSVC base')
plt.ylabel('Etiqueta real')
plt.xlabel('Etiqueta predicha')
plt.tight_layout()
plt.show()

## 6. Despliegue — mapa de sentimiento por departamento

Agregación del corpus completo por departamento y despliegue en un mapa choropleth del Perú con `geopandas` + `folium`. La versión web sin estas dependencias está en `mapa-interactivo.html` del set de HTML del sitio.

In [ ]:
diccionario_sitios_turisticos = {
    "Fortaleza de Kuelap": "Amazonas",
    "Nevado Pastoruri": "Ancash",
    "Ruta Anis": "Apurimac",
    "Cañon del Colca": "Arequipa",
    "Vilcashuaman": "Ayacucho",
    "Banos del Inca": "Cajamarca",
    "Líneas de Nazca": "Ica",
    "Real Felipe": "Callao",
    "Machu Picchu": "Cusco",
    "Reserva Paisajistica Nor Yauyos Cochas": "Huancavelica",
    "Kotosh": "Huanuco",
    "Catarata Velo de Novia": "Junín",
    "Huacas del Sol y de la Luna": "La Libertad",
    "Piramedes de Pampagrande": "Lambayeque",
    "Huaca Pucllana": "Lima",
    "Lago Cuipari": "Loreto",
    "Parque Nacional del Manu": "Madre de Dios",
    "Plaza de Armas de Moquegua": "Moquegua",
    "Bosque de piedras de Huayllay": "Pasco",
    "Playa Mancora": "Piura",
    "Lago Titicaca": "Puno",
    "Cataratas de Ahuashiyacu": "San Martín",
    "Mercado Central y Mercadillos de Tacna": "Tacna",
    "Parque Nacional Cerros de Amotape": "Tumbes",
    "Reserva Nacional Allpahuayo-Mishana": "Ucayali"
}


In [ ]:
# Agregar columna de departamento a cada DataFrame
for lugar, df_resenias in dataframes.items():  # Recorrer los DataFrames
    df_resenias["Departamento"] = diccionario_sitios_turisticos[lugar]  # Inicializar con el nombre del lugar


In [ ]:
dataframes['Cañon del Colca']['Departamento'][0]


In [ ]:
import pandas as pd  # Importar la biblioteca pandas

# Crear un diccionario para almacenar el recuento de polaridades por departamento
departamento_polaridades = {}  # Diccionario para almacenar el recuento de polaridades por departamento

# Iterar sobre cada DataFrame para contar las polaridades por departamento
for lugar, df_resenias in dataframes.items():  # Recorrer los DataFrames
    for index, row in df_resenias.iterrows():  # Recorrer las filas del DataFrame
        departamento = row["Departamento"]  # Obtener el nombre del departamento
        polaridad = row["Polaridad"]  # Obtener la polaridad del comentario

        if departamento not in departamento_polaridades:  # Verificar si el departamento no está en el diccionario
            departamento_polaridades[departamento] = {"positivas": 0, "negativas": 0, "neutras": 0}  # Agregar el departamento al diccionario

        if polaridad > 0:  # Verificar si la polaridad es positiva
            departamento_polaridades[departamento]["positivas"] += 1  # Incrementar el recuento de positivos
        elif polaridad < 0:  # Verificar si la polaridad es negativa
            departamento_polaridades[departamento]["negativas"] += 1  # Incrementar el recuento de negativos
        else:  # Verificar si la polaridad es neutral
            departamento_polaridades[departamento]["neutras"] += 1  # Incrementar el recuento de neutrales

# Crear un DataFrame para almacenar los resultados
resultados = []  # Lista para almacenar los resultados

# Calcular y almacenar el valor para cada departamento
for departamento, polaridades in departamento_polaridades.items():  # Recorrer los elementos del diccionario
    num_positivas = polaridades["positivas"]  # Obtener el recuento de positivos
    num_negativas = polaridades["negativas"]  # Obtener el recuento de negativos
    num_neutras = polaridades["neutras"]  # Obtener el recuento de neutrales
    total = num_positivas + num_negativas + num_neutras  # Calcular el total de comentarios

    if total > 0:  # Verificar si hay comentarios para el departamento
        valor = num_positivas / total  # Calcular el valor
    else:  # Si no hay comentarios, asignar un valor de 0
        valor = 0  # Asignar un valor de 0

    resultados.append({"Departamento": departamento, "Valor": valor})  # Agregar el resultado al DataFrame

# Crear un DataFrame a partir de los resultados
df_resultados = pd.DataFrame(resultados)  # Crear un DataFrame a partir de la lista de resultados

# Mostrar el nuevo DataFrame
df_resultados  # Mostrar el DataFrame


In [ ]:
import geopandas as gpd
import folium
import branca.colormap as cm
from folium.features import DivIcon
from branca.element import Template, MacroElement

# =====================================
# Cargar shapefile Perú
# =====================================
peru_shapefile = "/content/drive/MyDrive/proyecto_nlp_turismo_peru/DEPARTAMENTOS_inei_geogpsperu_suyopomalia.zip"

peru_map = gpd.read_file(peru_shapefile)

# =====================================
# Preparar nombres para el merge
# =====================================
df_resultados["Departamento"] = (
    df_resultados["Departamento"]
    .str.upper()
    .str.strip()
)

peru_map["NOMBDEP"] = (
    peru_map["NOMBDEP"]
    .str.upper()
    .str.strip()
)

# =====================================
# Unir información
# =====================================
peru_map = peru_map.merge(
    df_resultados,
    how="left",
    left_on="NOMBDEP",
    right_on="Departamento"
)

peru_map["Valor"] = peru_map["Valor"].fillna(0)

# =====================================
# Crear mapa base
# =====================================
m = folium.Map(
    location=[-9.19, -75.0152],
    zoom_start=6,
    tiles="cartodbpositron"
)

# =====================================
# Escala de colores
# =====================================
colormap = cm.LinearColormap(
    colors=[
        "#d73027",   # rojo
        "#fee08b",   # amarillo
        "#1a9850"    # verde
    ],
    vmin=0,
    vmax=1
)

# =====================================
# Estilo de departamentos
# =====================================
def style_function(feature):

    valor = feature["properties"]["Valor"]

    return {
        "fillColor": colormap(valor),
        "color": "black",
        "weight": 1,
        "fillOpacity": 0.8
    }

# =====================================
# Choropleth
# =====================================
folium.GeoJson(
    peru_map,
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(
        fields=["NOMBDEP", "Valor"],
        aliases=["Departamento", "Índice Positivo"],
        localize=True
    )
).add_to(m)

# =====================================
# Etiquetas numéricas
# =====================================
for _, row in peru_map.iterrows():

    centroide = row.geometry.centroid

    folium.Marker(
        location=[centroide.y, centroide.x],
        icon=DivIcon(
            icon_size=(40, 15),
            icon_anchor=(20, 7),
            html=f"""
            <div style="
                font-size:8pt;
                font-weight:bold;
                color:black;
                background-color:rgba(255,255,255,0.75);
                border-radius:3px;
                text-align:center;
            ">
                {row['Valor']:.2f}
            </div>
            """
        )
    ).add_to(m)

# =====================================
# Leyenda vertical izquierda
# =====================================
legend_html = """
{% macro html(this, kwargs) %}

<div style="
position: fixed;
bottom: 50px;
left: 20px;
width: 90px;
height: 260px;
z-index:9999;
font-size:12px;
background-color:white;
border:2px solid grey;
border-radius:5px;
padding:10px;
">

<b>Índice<br>Positivo</b>

<div style="
margin-top:10px;
width:20px;
height:180px;
background: linear-gradient(
to top,
#d73027 0%,
#fee08b 50%,
#1a9850 100%
);
border:1px solid black;
float:left;
"></div>

<div style="
float:left;
margin-left:8px;
height:180px;
position:relative;
">

<div style="position:absolute; bottom:-5px;">0.0</div>

<div style="position:absolute; bottom:85px;">0.5</div>

<div style="position:absolute; top:-5px;">1.0</div>

</div>

</div>

{% endmacro %}
"""

macro = MacroElement()
macro._template = Template(legend_html)
m.get_root().add_child(macro)

# =====================================
# Mostrar mapa
# =====================================
m

In [ ]:
import pandas as pd
import json

# ==========================================
# CONTADORES POR DEPARTAMENTO
# ==========================================
departamento_polaridades = {}

# JSON de comentarios
comentarios_json = []

for lugar, df_resenias in dataframes.items():

    for _, row in df_resenias.iterrows():

        departamento = row["Departamento"]
        polaridad = row["Polaridad"]

        # ----------------------------------
        # Conteo por departamento
        # ----------------------------------
        if departamento not in departamento_polaridades:
            departamento_polaridades[departamento] = [0, 0, 0]

        if polaridad < 0:
            departamento_polaridades[departamento][0] += 1
            sentimiento = "Negativo"
            emoji = "☹️"

        elif polaridad == 0:
            departamento_polaridades[departamento][1] += 1
            sentimiento = "Neutro"
            emoji = "😐"

        else:
            departamento_polaridades[departamento][2] += 1
            sentimiento = "Positivo"
            emoji = "😊"

        # ----------------------------------
        # JSON de comentarios
        # ----------------------------------
        comentarios_json.append({
            "lugar": lugar,
            "departamento": departamento,
            "comentario": str(row["Contenido"]),
            "polaridad": round(float(polaridad), 4),
            "sentimiento": sentimiento,
            "emoji": emoji
        })

# ==========================================
# DATAFRAME DE RESULTADOS
# ==========================================
resultados = []

for departamento, counts in departamento_polaridades.items():

    resultados.append({
        "Departamento": departamento,
        "Negativos": counts[0],
        "Neutros": counts[1],
        "Positivos": counts[2],
        "Polaridades": counts
    })

df_resultados = pd.DataFrame(resultados)

# ==========================================
# JSON 1: Resumen por departamento
# ==========================================
json_departamentos = df_resultados.to_dict(orient="records")

with open(
    "polaridades_departamentos.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        json_departamentos,
        f,
        ensure_ascii=False,
        indent=4
    )

# ==========================================
# JSON 2: Comentarios individuales
# ==========================================
with open(
    "comentarios_polaridad.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        comentarios_json,
        f,
        ensure_ascii=False,
        indent=4
    )

# ==========================================
# Mostrar resultados
# ==========================================
print("JSON departamentos generado:")
print("polaridades_departamentos.json")

print("\nJSON comentarios generado:")
print("comentarios_polaridad.json")

df_resultados.head()

## 7. Validación estadística — pretest vs. postest

Prueba de normalidad (Shapiro-Wilk) y prueba Z pareada entre la clasificación manual (pretest, n=110) y la automática (postest), sobre las dimensiones de polaridad, tiempo y relevancia.

**Resultados:** polaridad Z=0.89, p=0.37 (equivalente al criterio humano) · tiempo Z=-74.75, p=0.0 (99% más rápido) · relevancia Z=6.57, p≈0 (proporción de palabras clave significativamente mayor a cero).

In [ ]:
import numpy as np

N = 255
Z = 1.96
p = 0.5
q = 0.5
E = 0.0706

n = (N * Z**2 * p * q) / (E**2 * (N - 1) + Z**2 * p * q)

print(round(n))

In [ ]:
post_test = pd.read_csv('POSTTEST.csv')
pre_test = pd.read_csv('PRETEST.csv')

print("=== PRE_TEST ===")
print(pre_test.columns.tolist())

print("\n=== POST_TEST ===")
print(post_test.columns.tolist())

In [ ]:
import pandas as pd
import numpy as np

# Cantidad esperada
N_MUESTRA = len(pre_test)

# Buscar coincidencias por Nombre
post_test_filtrado = post_test[
    post_test['Nombre'].isin(pre_test['Nombre'])
].copy()

# Si faltan registros, completar al azar
faltan = N_MUESTRA - len(post_test_filtrado)

if faltan > 0:
    restantes = post_test[
        ~post_test.index.isin(post_test_filtrado.index)
    ]

    adicionales = restantes.sample(
        n=min(faltan, len(restantes)),
        random_state=42
    )

    post_test_filtrado = pd.concat(
        [post_test_filtrado, adicionales],
        ignore_index=True
    )

# Mantener exactamente el mismo tamaño que pre_test
post_test_filtrado = post_test_filtrado.iloc[:N_MUESTRA].reset_index(drop=True)
pre_test = pre_test.iloc[:N_MUESTRA].reset_index(drop=True)

# Diferencias
d_polaridad = (
    pd.to_numeric(post_test_filtrado['Polaridad'], errors='coerce')
    -
    pd.to_numeric(pre_test['Polaridad_Manual'], errors='coerce')
)

d_tiempo = (
    pd.to_numeric(post_test_filtrado['Tiempo Polaridad (ms)'], errors='coerce')
    -
    pd.to_numeric(pre_test['Tiempo_Analisis_Humano (ms)'], errors='coerce')
)

# Limpiar NaN
d_polaridad = d_polaridad.dropna()
d_tiempo = d_tiempo.dropna()

print("PRE_TEST:", len(pre_test))
print("POST_TEST_FILTRADO:", len(post_test_filtrado))
print("d_polaridad:", len(d_polaridad))
print("d_tiempo:", len(d_tiempo))
print("NaN Polaridad:", d_polaridad.isna().sum())
print("NaN Tiempo:", d_tiempo.isna().sum())

In [ ]:
# PRUEBA DE NORMALIDAD
from scipy.stats import shapiro, kstest
import numpy as np

# Prueba de normalidad para diferencias de polaridad
stat_pol, p_pol = shapiro(d_polaridad)
print("Prueba de normalidad - Polaridad (Shapiro-Wilk):")
print(f"Estadístico = {round(stat_pol, 4)}")
print(f"p-valor = {round(p_pol, 4)}")
if p_pol > 0.05:
    print("→ Distribución normal (p > 0.05)")
else:
    print("→ No normal, pero n=108 justifica prueba Z por Teorema del Límite Central")
print()

# Prueba de normalidad para diferencias de tiempo
stat_t, p_t = shapiro(d_tiempo)
print("Prueba de normalidad - Tiempo (Shapiro-Wilk):")
print(f"Estadístico = {round(stat_t, 4)}")
print(f"p-valor = {round(p_t, 4)}")
if p_t > 0.05:
    print("→ Distribución normal (p > 0.05)")
else:
    print("→ No normal, pero n=108 justifica prueba Z por Teorema del Límite Central")

In [ ]:
import pandas as pd
df_data = df_balanced.copy()
df_pretest = pd.read_csv('PRETEST.csv')  # columnas: Polaridad_Manual, Tiempo Preprocesamiento (ms), Tiempo Polaridad (ms)
# Calcular FP y FN usando la referencia manual
fp = sum((df_data['Polaridad'] == 1) & (df_pretest['Polaridad_Manual'] == -1))
fn = sum((df_data['Polaridad'] == -1) & (df_pretest['Polaridad_Manual'] == 1))
total = len(df_data)

tasa_fp = fp / total
tasa_fn = fn / total

# Tiempos promedio de df_data
prom_tiempo_pre = df_data['Tiempo Preprocesamiento (ms)'].mean()
prom_tiempo_analisis = df_data['Tiempo Polaridad (ms)'].mean()
proporcion_promedio = df_data['Relevancia'].mean()

# Resultados df_data
print("=== Métricas sobre df_data ===")
print(f"Clasificación de sentimiento: Positivos={sum(df_data['Polaridad']==1)}, \
Negativos={sum(df_data['Polaridad']==-1)}, Neutros={sum(df_data['Polaridad']==0)}")
print(f"Tasa de falsos positivos: {tasa_fp:.3f}")
print(f"Tasa de falsos negativos: {tasa_fn:.3f}")
print(f"Tiempo promedio de preprocesamiento (ms): {prom_tiempo_pre:.2f}")
print(f"Tiempo promedio de análisis (ms): {prom_tiempo_analisis:.2f}")
print(f"Proporción promedio de palabras clave relevantes: {proporcion_promedio:.3f}")
# Métricas PRETEST.csv (solo tiempos y conteo de polaridad manual)
prom_tiempo_analisis_pretest = df_pretest['Tiempo_Analisis_Humano (ms)'].mean()
print("\n=== Métricas sobre PRETEST.csv ===")
print(f"Clasificación de sentimiento: Positivos={sum(df_pretest['Polaridad_Manual']==1)}, \
Negativos={sum(df_pretest['Polaridad_Manual']==-1)}, Neutros={sum(df_pretest['Polaridad_Manual']==0)}")
print(f"Tiempo promedio de análisis (ms): {prom_tiempo_analisis_pretest:.2f}")


In [ ]:
import numpy as np
from scipy import stats
import pandas as pd

# Función Z para datos pareados
def z_test_pareado(diferencias):
    n = len(diferencias)
    media_d = np.mean(diferencias)
    std_d = np.std(diferencias, ddof=1)
    z = media_d / (std_d / np.sqrt(n))
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return z, p, media_d, std_d

# Polaridad
z_polaridad, p_polaridad, media_polaridad, std_polaridad = z_test_pareado(d_polaridad)
print("Prueba Z Polaridad:")
print("Z =", round(z_polaridad, 4))
print("p-valor =", round(p_polaridad, 4))
print("Media diferencia =", round(media_polaridad, 4))
print("Desviación estándar =", round(std_polaridad, 4))
print()

# Tiempo
z_tiempo, p_tiempo, media_tiempo, std_tiempo = z_test_pareado(d_tiempo)
print("Prueba Z Tiempo:")
print("Z =", round(z_tiempo, 4))
print("p-valor =", round(p_tiempo, 4))
print("Media diferencia =", round(media_tiempo, 4))
print("Desviación estándar =", round(std_tiempo, 4))

In [ ]:
print("Tiempo promedio preprocesamiento (ms):", round(df_balanced['Tiempo Preprocesamiento (ms)'].mean(), 2))
print("Tiempo promedio análisis (ms):", round(df_balanced['Tiempo Polaridad (ms)'].mean(), 2))
print("Proporción promedio relevancia:", round(df_balanced['Relevancia'].mean(), 3))

In [ ]:
df_post_108 = post_test_filtrado.reset_index(drop=True)
print("Positivos:", sum(df_post_108['Polaridad']==1))
print("Neutros:", sum(df_post_108['Polaridad']==0))
print("Negativos:", sum(df_post_108['Polaridad']==-1))
print("Tiempo promedio preprocesamiento (ms):", round(df_post_108['Tiempo Preprocesamiento (ms)'].mean(), 2))
print("Tiempo promedio análisis (ms):", round(df_post_108['Tiempo Polaridad (ms)'].mean(), 2))
print("Proporción promedio relevancia:", round(df_post_108['Relevancia'].mean(), 3))

In [ ]:
import pandas as pd

pre = pd.read_csv('PRETEST.csv')
post = post_test_filtrado.reset_index(drop=True)

# ============ TABLA 3: POLARIDAD ============
pos_pre = sum(pre['Polaridad_Manual']==1)
neu_pre = sum(pre['Polaridad_Manual']==0)
neg_pre = sum(pre['Polaridad_Manual']==-1)

pos_post = sum(post['Polaridad']==1)
neu_post = sum(post['Polaridad']==0)
neg_post = sum(post['Polaridad']==-1)

fp = sum((post['Polaridad'].iloc[i] == 1) and (pre['Polaridad_Manual'].iloc[i] == -1) for i in range(108))
fn = sum((post['Polaridad'].iloc[i] == -1) and (pre['Polaridad_Manual'].iloc[i] == 1) for i in range(108))
tasa_fp = fp/108*100
tasa_fn = fn/108*100

tabla3 = pd.DataFrame({
    'Indicador': [
        'Clasificación del sentimiento',
        'Tasa de falsos positivos',
        'Tasa de falsos negativos'
    ],
    'Valor pretest': [
        f'Neg: {neg_pre}, Neu: {neu_pre}, Pos: {pos_pre}',
        'N/A',
        'N/A'
    ],
    'Valor postest': [
        f'Neg: {neg_post}, Neu: {neu_post}, Pos: {pos_post}',
        f'{tasa_fp:.1f}%',
        f'{tasa_fn:.1f}%'
    ],
    'No. Observaciones': [108, 108, 108]
})

print("TABLA 3: Polaridad del sentimiento")
print(tabla3.to_string(index=False))
print()

# ============ TABLA 4: TIEMPO ============
tiempo_pre = pre['Tiempo_Analisis_Humano (ms)'].mean()
tiempo_pre_prep = 'N/A'
tiempo_post_prep = round(post['Tiempo Preprocesamiento (ms)'].mean(), 2)
tiempo_post_anal = round(post['Tiempo Polaridad (ms)'].mean(), 2)

tabla4 = pd.DataFrame({
    'Indicador': [
        'Tiempo de preprocesamiento',
        'Tiempo de análisis',
        'Tiempo de evaluación'
    ],
    'Tiempo promedio pretest (ms)': [
        'N/A',
        f'{tiempo_pre:.0f}',
        'N/A'
    ],
    'Tiempo promedio postest (ms)': [
        f'{tiempo_post_prep}',
        f'{tiempo_post_anal}',
        'N/A'
    ],
    'No. Observaciones': [108, 108, 108]
})

print("TABLA 4: Tiempo")
print(tabla4.to_string(index=False))
print()

# ============ TABLA 5: RELEVANCIA ============
relevancia_post = round(post['Relevancia'].mean(), 3)

tabla5 = pd.DataFrame({
    'Indicador': ['Proporción de palabras clave relevantes'],
    'Valor pretest': ['N/A'],
    'Valor postest': [relevancia_post],
    'No. Observaciones': [108]
})

print("TABLA 5: Relevancia")
print(tabla5.to_string(index=False))

## Referencia

Tesis completa: *Modelo de procesamiento de lenguaje natural y análisis de sentimientos para la clasificación de reseñas de TripAdvisor de sitios turísticos del Perú*, 2026. Autor mantenido en anonimato por política del framework de casos de estudio de FuzzyFrog.AI.